# Лабораторная работа №1. Transfer learning с ResNet на CIFAR-10

**Цель:** построить воспроизводимое сравнение трёх стратегий переноса обучения, а не получить максимально возможную метрику любой ценой.

После работы обучающийся сможет:

1. заменить классификационную голову ImageNet-модели под 10 классов CIFAR-10;
2. реализовать и объяснить **linear probing**, **частичное** и **полное** дообучение;
3. сравнить стратегии на одном фиксированном разбиении по accuracy, macro F1, времени и числу обучаемых параметров;
4. выбирать конфигурацию только по validation и один раз оценивать её на test;
5. воспроизвести запуск по seed, конфигурации и журналу артефактов.

**Бюджет:** smoke-режим — CPU, до 64/32/32 изображений и одна эпоха; полный режим — один GPU, 3–8 эпох. Обучение по умолчанию выключено: сначала завершите TODO и public checks.

**Критерий завершения:** три сопоставимых запуска, таблица метрик, confusion matrix лучшей по validation модели и анализ не менее пяти ошибок.

## 1. Теоретическая опора

ResNet обучает остаточное отображение (F(x)) и передаёт сигнал через shortcut: (y=F(x)+x). Это облегчает оптимизацию глубоких сетей. Предобученные на ImageNet признаки можно переносить на новую задачу тремя способами:

- **linear probing:** backbone заморожен, обучается только новая линейная голова;
- **partial fine-tuning:** разморожены голова и последняя стадия `layer4`;
- **full fine-tuning:** обновляются все параметры, обычно с меньшим learning rate для backbone.

Все три стратегии должны использовать одинаковые split, препроцессинг, seed, число эпох и правило выбора checkpoint. Test не участвует в подборе режима или гиперпараметров.

Используется актуальный API torchvision: `ResNet18_Weights.DEFAULT`. Объект весов предоставляет согласованный с ImageNet препроцессинг через `weights.transforms()`; внешний файл с именами классов не нужен.

## 2. Контракт эксперимента

Фиксируем `SEED=42`. Из официального train CIFAR-10 детерминированно выделяем train и validation, официальный test сохраняем для финальной оценки. В smoke-режиме берём детерминированные подмножества и выполняем весь pipeline на CPU.

Менять между строками основной таблицы разрешено только стратегию переноса. Если вы добавляете аугментации или меняете optimizer, оформите это как отдельную контролируемую серию.

Артефакты каждой строки: `config.json`, `history.json`, лучший checkpoint по validation, `metrics.json`. Итоговые артефакты: `summary.csv`, confusion matrix, пять разобранных ошибок и краткий отчёт.

## 3. Окружение и воспроизводимость

Используйте окружение репозитория. Переменные:

- `CV_SMOKE=1` — короткий CPU-проход (значение по умолчанию);
- `CV_RUN_TRAINING=1` — разрешить обучение после заполнения TODO;
- `CV_PUBLIC_CHECKS=1` — запустить открытые проверки контрактов.

In [ ]:
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
NUM_CLASSES = 10
CLASS_NAMES = ("airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck")
SMOKE_MODE = os.getenv("CV_SMOKE", "1") == "1"
RUN_TRAINING = os.getenv("CV_RUN_TRAINING", "0") == "1"
RUN_PUBLIC_CHECKS = os.getenv("CV_PUBLIC_CHECKS", "0") == "1"
DEVICE = torch.device("cpu" if SMOKE_MODE else ("cuda" if torch.cuda.is_available() else "cpu"))
DATA_ROOT = Path("data")
OUTPUT_ROOT = Path("outputs/resnet_transfer")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed()
print({"device": str(DEVICE), "smoke": SMOKE_MODE, "training_enabled": RUN_TRAINING})

## 4. Данные и фиксированное разбиение

Не создавайте новый случайный split для каждой стратегии. Индексы ниже формируются один раз и сохраняются в `split_manifest.json`. Для чистоты сравнения базовая серия использует один и тот же `weights.transforms()` без дополнительных аугментаций.

In [ ]:
weights = ResNet18_Weights.DEFAULT
EVAL_TRANSFORM = weights.transforms()

train_source = CIFAR10(root=DATA_ROOT, train=True, download=True, transform=EVAL_TRANSFORM)
test_source = CIFAR10(root=DATA_ROOT, train=False, download=True, transform=EVAL_TRANSFORM)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_source), generator=generator).tolist()

if SMOKE_MODE:
    train_indices = permutation[:64]
    val_indices = permutation[64:96]
    test_indices = list(range(32))
    batch_size = 8
else:
    val_indices = permutation[:5000]
    train_indices = permutation[5000:]
    test_indices = list(range(len(test_source)))
    batch_size = 128

split_manifest = {
    "seed": SEED,
    "train_indices": train_indices,
    "val_indices": val_indices,
    "test_indices": test_indices,
}
(OUTPUT_ROOT / "split_manifest.json").write_text(
    json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

train_dataset = Subset(train_source, train_indices)
val_dataset = Subset(train_source, val_indices)
test_dataset = Subset(test_source, test_indices)

def make_loader(dataset, *, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0 if SMOKE_MODE else 2,
        generator=torch.Generator().manual_seed(SEED),
    )

loaders = {
    "train": make_loader(train_dataset, shuffle=True),
    "val": make_loader(val_dataset, shuffle=False),
    "test": make_loader(test_dataset, shuffle=False),
}
print({name: len(loader.dataset) for name, loader in loaders.items()})

## 5. Модель и стратегии переноса

Голова ResNet-18 уже заменяется на слой с десятью выходами. Ваша ключевая реализация — функция `configure_transfer_learning`.

Контракт:

- `linear_probe`: обучаемые параметры есть только в `fc`;
- `partial_finetune`: обучаемы `layer4` и `fc`, остальные стадии заморожены;
- `full_finetune`: обучаемы все параметры;
- функция возвращает ту же модель и не меняет её архитектуру.

In [ ]:
TRANSFER_MODES = ("linear_probe", "partial_finetune", "full_finetune")

def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def configure_transfer_learning(model: nn.Module, mode: str) -> nn.Module:
    """Настройте requires_grad согласно контракту из markdown-ячейки."""
    if mode not in TRANSFER_MODES:
        raise ValueError(f"Неизвестный режим: {mode}")

    # TODO 1: сначала задайте единое состояние requires_grad для всех параметров.
    # TODO 2: разморозьте ровно те модули, которые нужны выбранному mode.
    # TODO 3: верните model. Не создавайте здесь новый экземпляр модели.
    raise NotImplementedError("Реализуйте три стратегии transfer learning")

def trainable_parameter_summary(model: nn.Module) -> dict:
    trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    return {
        "trainable_names": trainable,
        "trainable_count": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_count": sum(p.numel() for p in model.parameters()),
    }

probe_model = build_model()
with torch.inference_mode():
    probe_logits = probe_model(torch.zeros(2, 3, 224, 224))
assert probe_logits.shape == (2, NUM_CLASSES)
del probe_model, probe_logits

## 6. Обучение и оценка

Служебные циклы даны готовыми, чтобы фокус оставался на стратегии переноса и корректном эксперименте. Реализуйте optimizer с двумя группами параметров: для предобученной части используйте меньший learning rate, для новой головы — больший. В linear probe оптимизатор не должен получать замороженные параметры.

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    mode: str
    epochs: int
    backbone_lr: float
    head_lr: float
    weight_decay: float = 1e-4

def build_optimizer(model: nn.Module, config: ExperimentConfig) -> torch.optim.Optimizer:
    """Верните optimizer только для параметров с requires_grad=True."""
    # TODO 4: создайте группы backbone/head и назначьте им разные learning rate.
    # Подсказка: параметры головы доступны как model.fc.parameters().
    raise NotImplementedError("Соберите optimizer с дифференциальным learning rate")

def train_one_epoch(model, loader, optimizer, criterion) -> float:
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

@torch.inference_mode()
def predict(model, loader):
    model.eval()
    targets_all, predictions_all = [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE))
        predictions_all.extend(logits.argmax(dim=1).cpu().tolist())
        targets_all.extend(targets.tolist())
    return np.asarray(targets_all), np.asarray(predictions_all)

def classification_metrics(targets, predictions) -> dict:
    return {
        "accuracy": float(accuracy_score(targets, predictions)),
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
    }

def run_experiment(config: ExperimentConfig) -> dict:
    """Обучите одну конфигурацию, выбирая checkpoint только по validation macro F1."""
    # TODO 5:
    # 1) повторно зафиксируйте seed; 2) build_model; 3) configure_transfer_learning;
    # 4) build_optimizer; 5) обучайте одинаковое число эпох;
    # 6) сохраняйте лучший state_dict по VAL, не по TEST;
    # 7) после выбора checkpoint один раз оцените TEST;
    # 8) сохраните config/history/metrics в OUTPUT_ROOT/config.name.
    # Верните плоский словарь минимум с name, mode, val_accuracy, val_macro_f1,
    # test_accuracy, test_macro_f1, trainable_count, seconds.
    raise NotImplementedError("Соберите воспроизводимый запуск по контракту")

## 7. Сопоставимая серия

Не меняйте бюджеты между тремя строками. Значения learning rate можно уточнить по validation, но test остаётся закрытым до окончательного выбора. В отчёте обязательно укажите, если один seed и короткий бюджет не позволяют делать устойчивые выводы.

In [ ]:
EPOCHS = 1 if SMOKE_MODE else 5
EXPERIMENTS = [
    ExperimentConfig("linear_probe", "linear_probe", EPOCHS, backbone_lr=0.0, head_lr=1e-3),
    ExperimentConfig("partial_finetune", "partial_finetune", EPOCHS, backbone_lr=1e-4, head_lr=1e-3),
    ExperimentConfig("full_finetune", "full_finetune", EPOCHS, backbone_lr=1e-5, head_lr=1e-3),
]

results = []
if RUN_TRAINING:
    for experiment in EXPERIMENTS:
        results.append(run_experiment(experiment))
    summary = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
    summary.to_csv(OUTPUT_ROOT / "summary.csv", index=False)
    display(summary)
else:
    print("Обучение выключено. Завершите TODO и задайте CV_RUN_TRAINING=1.")

## 8. Открытые проверки

Public checks видимы студенту и проверяют только контракт, не качество на скрытом наборе. Они намеренно не содержат эталонной реализации. После заполнения TODO запустите с `CV_PUBLIC_CHECKS=1`.

In [ ]:
def run_public_checks() -> None:
    assert SEED == 42
    assert not set(train_indices) & set(val_indices), "Train и validation пересекаются"
    assert len(CLASS_NAMES) == NUM_CLASSES == 10
    assert tuple(build_model()(torch.zeros(2, 3, 224, 224)).shape) == (2, 10)

    for mode in TRANSFER_MODES:
        model = configure_transfer_learning(build_model(), mode)
        info = trainable_parameter_summary(model)
        names = info["trainable_names"]
        assert names, f"{mode}: нет обучаемых параметров"
        if mode == "linear_probe":
            assert all(name.startswith("fc.") for name in names)
        elif mode == "partial_finetune":
            assert all(name.startswith(("layer4.", "fc.")) for name in names)
            assert any(name.startswith("layer4.") for name in names)
        else:
            assert info["trainable_count"] == info["total_count"]

    print("Public checks: OK")

if RUN_PUBLIC_CHECKS:
    run_public_checks()
else:
    print("Public checks определены; включение: CV_PUBLIC_CHECKS=1.")

## 9. Анализ и сдаваемые артефакты

Заполните после эксперимента:

1. **Таблица:** validation/test accuracy и macro F1, время, доля обучаемых параметров.
2. **Матрица ошибок:** для лучшей по validation конфигурации.
3. **Пять ошибок:** изображение, истинный/предсказанный класс, наблюдение и правдоподобная причина.
4. **Вывод:** какая стратегия оправдана при данном бюджете; не обобщайте результат за пределы CIFAR-10, одного split и seed.
5. **Воспроизводимость:** версии Python/PyTorch/torchvision, устройство, seed, commit, значения конфигурации.

Работа не засчитывается, если режим выбран по test, стратегии используют разные split/число эпох, отсутствует `split_manifest.json` или приведено только лучшее число без журнала запусков.

### Контрольные вопросы

1. Почему linear probe измеряет пригодность уже выученного пространства признаков?
2. Почему для случайно инициализированной головы обычно нужен больший learning rate?
3. Что меняется в BatchNorm при `model.train()`, даже если параметры заморожены?
4. Почему один seed не позволяет утверждать, что небольшое отличие стратегий устойчиво?
5. Какие признаки domain shift возникают при переносе ImageNet → CIFAR-10?